In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import numpy as np
import datetime
import time

import sys
sys.path.append("src")

from data_generation import *
from neural_networks import *
from estimation import *

pd.options.mode.chained_assignment = None
os.chdir('Result_Tables/')

## Table 3

In [2]:
seed_list = [7456,145609, 214341, 237639, 148649, 763490, 677253, 560404, 591830,
    84947,  26672, 184535, 874369,  39876, 738836, 100589, 812811,
    75122, 182935, 911677]

dg_list = [rcl_log, rcl_sin] ## non-linear data generation
JKM_list = [[10,0,100]]  # only price feature is included
delta = 0.1 

hyper_params = {'hidden_size': 256, 'num_hidden_layers': 5, 'n_epochs': 2000, 'learning_rate': 0.0001}

for [J, K, M] in JKM_list:
    for dg in dg_list:
        for i in range(len(seed_list[1:3])): ### this was changed to 3 for illustration.
            print(J, K, M, i, 'started', datetime.datetime.now())
            seed = seed_list[i]
            params = [np.random.normal(0,1,K+2) , np.ones(K+2)]
            params[0][-1] = -1
            full_one_iteration(J, M, K, seed, dg, params, hyper_params, delta)   ### this line runs for about 5-10 min 
            print(J, K, M, i, 'finished', datetime.datetime.now())
        report(J, K, M, seed_list[1:3], dg)

10 0 100 0 started 2026-03-09 15:39:38.207860
Data is generated.
Main model is trained.
RCL is estimated.
MNL is estimated.
NP is estimated.
Prediction errors are calculated.
Elasticity errors are calculated.
10 0 100 0 finished 2026-03-09 15:41:47.775222
10 0 100 1 started 2026-03-09 15:41:47.775263
Data is generated.
Main model is trained.
RCL is estimated.
MNL is estimated.
NP is estimated.
Prediction errors are calculated.
Elasticity errors are calculated.
10 0 100 1 finished 2026-03-09 15:43:50.878651
10 0 100 0 started 2026-03-09 15:43:51.456978
Data is generated.
Main model is trained.
RCL is estimated.
MNL is estimated.
NP is estimated.
Prediction errors are calculated.
Elasticity errors are calculated.
10 0 100 0 finished 2026-03-09 15:45:54.429237
10 0 100 1 started 2026-03-09 15:45:54.429275
Data is generated.
Main model is trained.
RCL is estimated.
MNL is estimated.
NP is estimated.
Prediction errors are calculated.
Elasticity errors are calculated.
10 0 100 1 finished 202

In [4]:
## this step reads the middle tables saved from the above cell
JKM_dg_list = [[10,0,100,rcl_log],[10,0,100,rcl_sin]]
elas_median_df, elas_mae_df, elas_rmse_df, error_df  = output(JKM_dg_list)

In [5]:
error_rslt = error_df.groupby(['dg','J','M','K'])[['mae_deep','mae_mnl','mae_rcl','mae_single','mae_random',
                                                  'mse_deep','mse_mnl','mse_rcl','mse_single','mse_random']].mean().reset_index()
error_rslt[['rmse_deep','rmse_mnl','rmse_rcl','rmse_single','rmse_random']] = np.sqrt(error_rslt[['mse_deep','mse_mnl','mse_rcl','mse_single','mse_random']])
error_rslt['obs'] = error_rslt['M'] * error_rslt['J'] *  0.2
error_rslt = error_rslt[[ 'dg', 'J', 'M', 'K', 
                             'mae_deep', 'rmse_deep', 
                             'mae_mnl', 'rmse_mnl', 
                             'mae_rcl', 'rmse_rcl',  
                             'mae_single', 'rmse_single', 
                             'mae_random','rmse_random',
                             'obs']]
error_rslt

,dg,J,M,K,mae_deep,rmse_deep,mae_mnl,rmse_mnl,mae_rcl,rmse_rcl,mae_single,rmse_single,mae_random,rmse_random,obs
0,rcl_log,10,100,0,0.002879,0.037638,0.035057,0.132180,0.025270,0.101676,0.095070,0.194228,0.104945,0.215056,200.0
1,rcl_sin,10,100,0,0.002332,0.020891,0.029138,0.119628,0.011325,0.063677,0.038955,0.129192,0.045340,0.143167,200.0


In [6]:
elas_rmse_df.columns = [col.replace('ae_', 'rmse_') if 'ae_' in col else col for col in elas_rmse_df.columns]
elas_mae_df.columns = [col.replace('ae_', 'mae_') if 'ae_' in col else col for col in elas_mae_df.columns]

elas_rslt = pd.merge(elas_mae_df, elas_rmse_df,
                     on = ['type','dg','J','M','K'])
elas_rslt['obs'] = (elas_rslt['M'] * elas_rslt['J'] * 20 * 0.8).astype('int')
elas_rslt.loc[elas_rslt.type == 'cross','obs'] = (elas_rslt['M'] * elas_rslt['J'] * 20 * (elas_rslt['J']-1) * 0.8).astype('int')

elas_rslt = elas_rslt[['type', 'dg', 'J', 'M', 'K', 
                             'mae_deep', 'rmse_deep', 
                             'mae_mnl', 'rmse_mnl', 
                             'mae_rcl', 'rmse_rcl',  
                             'mae_single', 'rmse_single', 
                             'obs']]
elas_rslt

,type,dg,J,M,K,mae_deep,rmse_deep,mae_mnl,rmse_mnl,mae_rcl,rmse_rcl,mae_single,rmse_single,obs
0,cross,rcl_log,10,100,0,0.027288,0.052152,0.083300,0.155778,0.166111,0.435077,0.428650,0.632445,144000
1,self,rcl_log,10,100,0,0.090396,0.134502,4.738121,5.416897,1.596513,2.138048,0.797300,1.045792,16000
2,cross,rcl_sin,10,100,0,0.040623,0.084173,0.098527,0.182418,0.093236,0.177504,0.238026,0.352579,144000
3,self,rcl_sin,10,100,0,0.127097,0.274511,0.941857,1.215872,0.280994,0.410039,0.883518,1.273454,16000


## Table 4

In [7]:
seed_list = [7456,145609, 214341, 237639, 148649, 763490, 677253, 560404, 591830,
    84947,  26672, 184535, 874369,  39876, 738836, 100589, 812811,
    75122, 182935, 911677]

dg_list = [rcl_in3] ## consumer with inattention
JKM_list = [[2,0,100],[5,0,100],[10,0,100]]  # only price feature is included
delta = 0.1 

hyper_params = {'hidden_size': 256, 'num_hidden_layers': 5, 'n_epochs': 2000, 'learning_rate': 0.0001}

for [J, K, M] in JKM_list:
    for dg in dg_list:
        for i in range(len(seed_list[1:3])):
            print(J, K, M, i, 'started', datetime.datetime.now())
            seed = seed_list[i]
            params = [np.random.normal(0,1,K+2), np.ones(K+2)]
            params[0][-1] = -1
            full_one_iteration(J, M, K, seed, dg, params, hyper_params, delta)   ### this line runs for about 5-10 min 
            print(J, K, M, i, 'finished', datetime.datetime.now())
        report(J, K, M, seed_list, dg)

2 0 100 0 started 2026-03-09 15:50:29.857968
Data is generated.
Main model is trained.
RCL is estimated.
MNL is estimated.
NP is estimated.
Prediction errors are calculated.
Elasticity errors are calculated.
2 0 100 0 finished 2026-03-09 15:52:15.948238
2 0 100 1 started 2026-03-09 15:52:15.948278
Data is generated.
Main model is trained.
RCL is estimated.
MNL is estimated.
NP is estimated.
Prediction errors are calculated.
Elasticity errors are calculated.
2 0 100 1 finished 2026-03-09 15:53:58.487996
5 0 100 0 started 2026-03-09 15:53:58.741498
Data is generated.
Main model is trained.
RCL is estimated.
MNL is estimated.
NP is estimated.
Prediction errors are calculated.
Elasticity errors are calculated.
5 0 100 0 finished 2026-03-09 15:56:01.065745
5 0 100 1 started 2026-03-09 15:56:01.065785
Data is generated.
Main model is trained.
RCL is estimated.
MNL is estimated.
NP is estimated.
Prediction errors are calculated.
Elasticity errors are calculated.
5 0 100 1 finished 2026-03-09 

In [8]:
## this step reads the middle tables saved from the above cell
JKM_dg_list = [[2,0,100,rcl_in3], [5,0,100,rcl_in3], [10,0,100,rcl_in3]]
elas_median_df, elas_mae_df, elas_rmse_df, error_df  = output(JKM_dg_list)

In [9]:
error_rslt = error_df.groupby(['dg','J','M','K'])[['mae_deep','mae_mnl','mae_rcl','mae_single','mae_random',
                                                  'mse_deep','mse_mnl','mse_rcl','mse_single','mse_random']].mean().reset_index()
error_rslt[['rmse_deep','rmse_mnl','rmse_rcl','rmse_single','rmse_random']] = np.sqrt(error_rslt[['mse_deep','mse_mnl','mse_rcl','mse_single','mse_random']])
error_rslt['obs'] = error_rslt['M'] * error_rslt['J'] *  0.2
error_rslt = error_rslt[[ 'dg', 'J', 'M', 'K', 
                             'mae_deep', 'rmse_deep', 
                             'mae_mnl', 'rmse_mnl', 
                             'mae_rcl', 'rmse_rcl',  
                             'mae_single', 'rmse_single', 
                             'mae_random','rmse_random',
                             'obs']]
error_rslt

,dg,J,M,K,mae_deep,rmse_deep,mae_mnl,rmse_mnl,mae_rcl,rmse_rcl,mae_single,rmse_single,mae_random,rmse_random,obs
0,rcl_in3,2,100,0,0.005558,0.026836,0.058163,0.092939,0.031790,0.063928,0.007473,0.030488,0.177026,0.219460,40.0
1,rcl_in3,5,100,0,0.006743,0.023795,0.020257,0.046751,0.017840,0.043566,0.024874,0.053578,0.076750,0.119052,100.0
2,rcl_in3,10,100,0,0.003495,0.017382,0.013681,0.037383,0.007353,0.027731,0.026327,0.058871,0.048610,0.084994,200.0


In [10]:
elas_rmse_df.columns = [col.replace('ae_', 'rmse_') if 'ae_' in col else col for col in elas_rmse_df.columns]
elas_mae_df.columns = [col.replace('ae_', 'mae_') if 'ae_' in col else col for col in elas_mae_df.columns]

elas_rslt = pd.merge(elas_mae_df, elas_rmse_df,
                     on = ['type','dg','J','M','K'])
elas_rslt['obs'] = (elas_rslt['M'] * elas_rslt['J'] * 20 * 0.8).astype('int')
elas_rslt.loc[elas_rslt.type == 'cross','obs'] = (elas_rslt['M'] * elas_rslt['J'] * 20 * (elas_rslt['J']-1) * 0.8).astype('int')

elas_rslt = elas_rslt[['type', 'dg', 'J', 'M', 'K', 
                             'mae_deep', 'rmse_deep', 
                             'mae_mnl', 'rmse_mnl', 
                             'mae_rcl', 'rmse_rcl',  
                             'mae_single', 'rmse_single', 
                             'obs']]
elas_rslt

,type,dg,J,M,K,mae_deep,rmse_deep,mae_mnl,rmse_mnl,mae_rcl,rmse_rcl,mae_single,rmse_single,obs
0,cross,rcl_in3,2,100,0,0.569114,4.253520,1.900631,8.896219,2.236179,8.598612,1.030607,5.682758,3200
1,self,rcl_in3,2,100,0,0.163725,0.537911,1.025705,1.639955,0.848504,1.500376,0.255798,0.890199,3200
2,cross,rcl_in3,5,100,0,0.918861,4.770158,0.894595,5.873336,0.861177,5.882083,1.121770,5.584614,32000
3,self,rcl_in3,5,100,0,0.928046,1.941067,1.204885,1.981179,0.889834,1.989595,1.218926,1.991035,8000
4,cross,rcl_in3,10,100,0,0.496812,4.384343,0.461912,4.109623,0.425210,4.128616,0.652529,4.147515,144000
5,self,rcl_in3,10,100,0,1.412341,23.907937,1.435375,2.153274,0.901770,2.326774,1.358703,2.158163,16000
